# Paper visualization

- kernel: R 4.1.3 (r_env)
- desc: Figures used in the paper

## load modules

In [1]:
library(tidyverse)
library(ggsci)
library(patchwork)
library(ggpubr)
library(ggsci)
library(ggrepel)
library(eulerr)

theme_set(theme_pubr(12))
options(readr.show_col_types = FALSE)

source('../r_utils.r')

── Attaching packages ─────────────────────────────────────── tidyverse 1.3.2 ──
✔ ggplot2 3.3.6     ✔ purrr   0.3.4
✔ tibble  3.1.7     ✔ dplyr   1.0.9
✔ tidyr   1.2.0     ✔ stringr 1.4.0
✔ readr   2.1.2     ✔ forcats 0.5.1
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()


In [2]:
wkdir = '../figures/'

# clinical info
df_clin <- read_csv('../tables/clinical-info.csv') %>% 
    mutate(
        major_group = factor(major_group, levels = names(major_group_col)),  # Case = KBP000166 data, Ctrl = GD_Con data
        detail_group = factor(detail_group, levels = names(age_group_col)),
        gender = factor(gender, levels = names(gender_col))
    )
# taxon profile
df_profile <- read_csv('../tables/taxonomic_clin.csv') %>% 
    select('sample', 'clade_name', 'clade_short', 'NCBI_tax_id', 'clade_type', 'relab')

## clinical info

In [51]:
pdata <- df_clin

In [61]:
# Gender in dataset (major group)
p_gender <- pdata %>% 
    count(major_group, gender, name = 'cnt') %>%
    ggplot(aes(x=major_group, y=cnt, fill=gender, label=cnt)) +
    geom_bar(stat='identity', position = position_dodge()) +
    geom_text(aes(y=cnt + 1), position = position_dodge(width = 0.9)) +
    scale_fill_manual(values = c(gender_col, 'NA'='gray')) +
    labs(x='Dataset', y='Num. of Donors', fill='Gender')

In [62]:
# Age group by dataset (major group)
p_age <- pdata %>% 
    count(detail_group, major_group, name = 'cnt') %>%
    ggplot(aes(x=major_group, y=cnt, fill=detail_group, label=cnt)) +
    geom_bar(stat='identity', position = position_dodge()) +
    geom_text(aes(y=cnt + 1), position = position_dodge(width = 0.9)) +
    scale_fill_manual(values = age_group_col) +
    labs(x='Dataset', y='Num. of Donors', fill='Age Group')

In [65]:
# Gender in age group
p3 <- pdata %>% 
    count(gender, detail_group, name = 'cnt') %>%
    ggplot(aes(x=detail_group, y=cnt, fill=gender, label=cnt)) +
    geom_bar(stat='identity', position = position_dodge(), show.legend = F) +
    geom_text(aes(y=cnt + 0.5), position = position_dodge(width = 0.9), size=3) +
    scale_fill_manual(values = gender_col) +
    labs(x='Age Group', y='Num. of Donors', fill='Gender')

In [67]:
ggsave(
    filename = file.path(wkdir, 'barplot-clinical-info.pdf'), width=9, height=6,
    plot = wrap_plots((p_gender | p_age) / p3, guides = 'collect') & theme(legend.position = 'right')
)

## abundance DA at species level

In [336]:
status_col = c(change_col, 'unchange'='gray')
print(status_col)

p_threshold = 0.05

# Differential analysis results
df_res <- read_csv('../tables/abundance-da-species.csv') %>% 
    mutate(
        status = factor(status, levels = names(status_col))
    )

       up      down  unchange 
"#E41A1C" "#514fef"    "gray" 


### volcano show abundance change type: up/down/unchange

In [337]:
# volcano plot
pdata <- df_res
p <- ggplot(pdata, aes(x = log2fc, y = -log10(p.adj), fill = status, size = abs(diff))) + 
    geom_point(alpha = 0.7, shape = 21) + 
    geom_hline(yintercept = -log10(p_threshold), linetype = 'dashed', color = 'red') + 
    scale_fill_manual(values = status_col)  +
    labs(x='Log2(Fold Change)', y='-Log10(Adjusted P-value)', size='Abs(Diff)', fill='Abundance\nChange') +
    guides(fill=guide_legend(override.aes = list(size=3))) +
    theme(legend.position = 'right')
ggsave(filename = file.path(wkdir, 'volcano-abundance-da.pdf'), plot = p, width = 9, height = 6)

### boxplot show abundance change along age group

In [111]:
df <- df_res %>% 
    select(status, clade_name) %>% 
    merge(df_profile, by = 'clade_name') %>% 
    merge(df_clin, by = 'sample')

In [137]:
# for up results: N=14
pdata <- df %>% filter(status == 'up')
facet_bg_col = status_col[['up']]

p <- ggplot(data = pdata, aes(x = detail_group, y = relab)) + 
    geom_boxplot(outlier.shape = NA) +
    geom_point(
        aes(fill = major_group), 
        shape = 21, size = 1.5, alpha = 0.6,
        position = position_jitter(0.3)
    )+ 
    facet_wrap(. ~ clade_short, ncol = 3, scales = 'free') +
    scale_fill_manual(values = major_group_col) +
    guides(fill=guide_legend(override.aes = list(size=3))) +
    labs(x='Age Group', y='Relative Abundance', fill='Dataset') +
    theme(
        legend.position = 'top',
        strip.background = element_rect(fill = adjustcolor(facet_bg_col, alpha.f = 0.2)),
        axis.text.x = element_text(angle = 45, hjust = 0.9)
    ) 
ggsave(filename = file.path(wkdir, 'box-relab-along-age_group-up.pdf'), plot=p, width=8, height=12)

In [138]:
# for up results: N=14
pdata <- df %>% filter(status == 'down')
facet_bg_col = status_col[['down']]

p <- ggplot(data = pdata, aes(x = detail_group, y = relab)) + 
    geom_boxplot(outlier.shape = NA) +
    geom_point(
        aes(fill = major_group), 
        shape = 21, size = 1.5, alpha = 0.6,
        position = position_jitter(0.3)
    )+ 
    facet_wrap(. ~ clade_short, ncol = 3, scales = 'free') +
    scale_fill_manual(values = major_group_col) +
    guides(fill=guide_legend(override.aes = list(size=3))) +
    labs(x='Age Group', y='Relative Abundance', fill='Dataset') +
    theme(
        legend.position = 'top',
        strip.background = element_rect(fill = adjustcolor(facet_bg_col, alpha.f = 0.2)),
        axis.text.x = element_text(angle = 45, hjust = 0.9)
    ) 
ggsave(filename = file.path(wkdir, 'box-relab-along-age_group-down.pdf'), plot=p, width=8, height=7)

### simple linear results: barplot

In [197]:
pdata <- read_csv('../tables/abundance-slr-age.csv')
clade_order <- pdata %>%
    arrange(desc(da.status), desc(clade_short)) %>%
    pull(clade_short)
pdata <- pdata %>% 
    mutate(
        slr.status = stringr::str_to_title(slr.status),
        clade_short = factor(clade_short, levels = clade_order)
    )

color_map <- c(
    'Positive' = change_col[['up']],
    'Negative' = change_col[['down']],
    'Irrelevant' = 'gray'
)

In [198]:
p <- ggplot(data = pdata, aes(x=r.squared, y=clade_short, fill=slr.status)) +
    geom_bar(stat = 'identity') +
    labs(x='R-Squared', y='', fill='Correlation\nCategory') +
    scale_fill_manual(values = color_map) +
    theme(axis.title.y = element_blank())
ggsave(filename = file.path(wkdir, 'barplot-slr-r2.pdf'), plot = p, width = 8, height = 5)

### Multiple Linear Regression (mlr) results: barplot

In [11]:
valid_clades <- c(
    'Acidaminococcus_intestini', 'Anaerostipes_hadrus', 'Collinsella_aerofaciens',
    'Coprococcus_comes', 'Dialister_sp_CAG_357', 'Eubacterium_siraeum', 
    'Faecalibacterium_prausnitzii', 'Fusicatenibacter_saccharivorans', 'Ruminococcus_gnavus'
)
# data
df <- read_csv('../tables/abundance-mlr-results.csv')

#### barplot for aggregated Age Group

In [12]:
# color map
color_map <- c(
    'NS' = "gray70",
    'p < 0.05' = "#F781BF",
    'p < 0.01' = "#EE4C97"
)

df_process <- df %>% 
    mutate(vars = case_when(
        startsWith(x = vars, prefix = 'group') ~ 'age_group',
        TRUE ~ vars
    )) %>%
    filter(vars %in% c('genderM', 'age_group'))

In [13]:
pdata <- df_process %>% filter(vars == 'genderM')
# add age group results: use the result with lowest p
for (clade in unique(df_process$clade_short)) {
    subdf <- filter(df_process, clade_short == clade) %>%
        filter(vars == 'age_group')
    pdata <- rbind(pdata, subdf[which.min(subdf$p), ])
}
# signif label
clade_col <- pdata %>% 
    select(clade_short) %>% 
    unique() %>% 
    mutate(color = ifelse(clade_short %in% valid_clades, "#1782ae", "black")) %>%
    arrange(desc(clade_short)) %>%
    pull(color, name = clade_short)

pdata <- pdata %>% 
    mutate(signif = case_when(
        p < 0.01 ~ 'p < 0.01',
        p < 0.05 ~ 'p < 0.05',
        p < 1 ~ 'NS'
    )) %>%
    mutate(
        signif = factor(signif, levels = names(color_map)),
        facet_by = case_when(
            vars == 'genderM' ~ 'Gender (M)',
            vars == 'age_group' ~ 'Age Group'
        ),
        clade_short = factor(clade_short, levels = names(clade_col))
    )

In [14]:
p <- ggplot(data = pdata, aes(x=estimate, y=clade_short, fill=signif)) +
    geom_bar(stat = 'identity') +
    labs(x='Estimate', y='', fill='Significance') +
    scale_fill_manual(values = color_map) +
    facet_wrap(~ facet_by) +
    theme(
        strip.background = element_rect(fill=NA),
        axis.title.y = element_blank(),
        axis.text.y = element_text(color = clade_col)
    )
ggsave(filename = file.path(wkdir, 'barplot-mlr-signif.pdf'), plot = p, width = 9, height = 5)

Warning message:
“Vectorized input to `element_text()` is not officially supported.
Results may be unexpected or may change in future versions of ggplot2.”


#### DotPlot for all Age Groups

In [38]:
# shape map
shape_map <- c(
    'p < 0.05' = "circle filled",
    'NS' = "square filled"
)
# var names
varname_map <- c(
    'group[20, 40)' = 'Age Group\n[20, 40)',
    'group[40, 60)' = 'Age Group\n[40, 60)',
    'group[60, 80)' = 'Age Group\n[60, 80)',
    'group[80, 100)' = 'Age Group\n[80, 100)',
    'genderM' = 'Gender\n(M)'
)
pdata <- df %>% 
    filter(vars %in% names(varname_map)) %>% 
    mutate(
        varname = recode(vars, !!!varname_map)  # !!! similar to ** in python
    )

In [39]:
# tick texts colors
clade_col <- pdata %>% 
    select(clade_short) %>% 
    unique() %>% 
    mutate(color = ifelse(clade_short %in% valid_clades, "#1782ae", "black")) %>%
    arrange(desc(clade_short)) %>%
    pull(color, name = clade_short)

pdata <- pdata %>% 
    mutate(signif = case_when(
        p < 0.05 ~ 'p < 0.05',
        p < 1 ~ 'NS'
    )) %>%
    mutate(
        signif = factor(signif, levels = names(shape_map)),
        clade_short = factor(clade_short, levels = names(clade_col))
    )

In [40]:
p <- ggplot(data = pdata, aes(x=varname, y=clade_short, fill=estimate)) +
    geom_point(aes(shape=signif), size=5) +
    scale_fill_gradient2(low = 'darkblue', high = 'darkred', mid = 'white', midpoint = 0) +
    scale_shape_manual(values = shape_map) +
    labs(x='', y='', fill='Estimate', shape='Significance') +
    theme(
        strip.background = element_rect(fill=NA),
        axis.title = element_blank(),
        axis.text.y = element_text(color = clade_col),
        legend.position = 'right'
    )
ggsave(filename = file.path(wkdir, 'dotplot-mlr-signif.pdf'), plot = p, width = 9, height = 5)

Warning message:
“Vectorized input to `element_text()` is not officially supported.
Results may be unexpected or may change in future versions of ggplot2.”


## AMP analysis

### vlocano show AMP DA results: up/down/unchange

In [524]:
color_map = c(change_col, 'unchange'='gray')
p_threshold <- 0.01
log2fc_threshold <- 1

pdata <- read_csv('../tables/amp-da-results.csv') %>%
    mutate(
        status = factor(status, levels = names(color_map))
    )

In [525]:
p_da <- ggplot(pdata, aes(x = log2fc, y = -log10(p_adj), fill = status)) + 
    geom_point(alpha = 0.7, shape = 21, size = 1.5) + 
    geom_hline(yintercept = -log10(p_threshold), linetype = 'dashed', color = 'red') + 
    geom_vline(xintercept = c(log2fc_threshold, -log2fc_threshold), , linetype = 'dashed', color = 'red') +
    scale_fill_manual(values = color_map)  +
    labs(x='Log2(Fold Change)', y='-Log10(Adjusted P-value)', fill='Abundance\nChange') +
    guides(fill=guide_legend(override.aes = list(size=3), nrow = 2, byrow = F)) +
    theme(legend.position = 'top')
# ggsave(filename = file.path(wkdir, 'volcano-amp-da.pdf'), plot = p_da, width = 4, height = 4.5)

### vlocano show validation evidence

- for up/down AMPs

In [518]:
color_map <- c(
    'Experimentally Validated' = "#E41A1C",
    'Predicted' = "#377EB8",
    'Predicted (Based on signature)' = "gray"
)
pdata <- read_csv('../tables/amp-da-anno-evidence.csv') %>%
    mutate(
        validation = factor(validation, levels = names(color_map))
    ) %>%
    arrange(desc(validation))
ann_data <- pdata %>% 
    filter(validation == 'Experimentally Validated')

In [519]:
p_evidence <- ggplot(pdata, aes(x = log2fc, y = -log10(p_adj))) +
    geom_point(aes(fill = validation), size = 4, pch=21, alpha = 0.85) +
    geom_label_repel(min.segment.length = 0.0,
        data = ann_data, aes(label = paste0(camp, '\n', source_organism)), 
        box.padding = 1, fill = '#ffffff04', size = 3
    ) +
    scale_fill_manual(values = color_map) +
    guides(fill = guide_legend(nrow = 2, byrow = TRUE)) + 
    labs(x='Log2(Fold Change)', y='-Log10(Adjusted P-value)', fill='')
# ggsave(filename = file.path(wkdir, 'scatter-amp-evidence.pdf'), plot = p_evidence, width = 5, height = 5)

### AMP relative abundance of experimentally validated AMPs

In [520]:
expr_val_amp = c(
    'CAMPSQ67', 'CAMPSQ1136', 'CAMPSQ3690', 'CAMPSQ4486',  # up
    'CAMPSQ574', 'CAMPSQ3871'  # down
)

df_amp_profile <- read_tsv('../tables/amp-relab-profile.csv') %>%
    pivot_longer(cols = -c('camp'), names_to = 'sample', values_to = 'relab')

In [521]:
pdata <- df_amp_profile %>% 
    filter(camp %in% expr_val_amp) %>% 
    merge(df_clin, by = 'sample') %>% 
    mutate(camp = factor(camp, levels = expr_val_amp))

In [522]:
p_relab <- ggplot(data = pdata, aes(x = detail_group, y = relab)) +
    geom_violin(scale = 'width') +
    geom_jitter(aes(fill = major_group), size = 1.5, alpha = 0.75, shape = 21, width = 0.2) +
    facet_wrap( ~ camp, scales = 'free_y') +
    scale_fill_nejm() +
    labs(x = 'Age Group', y = 'AMP Relative Abundance', fill = 'Dataset') +
    theme(
        legend.position = 'right',
        strip.background = element_rect(fill = NA),
        axis.text.x = element_text(angle = 45, hjust = 0.95)
    )
# ggsave(filename = file.path(wkdir, 'vln-selected_amp-relab.pdf'), plot = p_relab, width = 9, height = 6)

### merge plots

In [526]:
# save togather
my_layout <- "
AAAABBBBBBB
CCCCCCCCCCC
"
ggsave(
    filename = file.path(wkdir, 'amp-analysis-result.pdf'), width = 10, height = 8,
    plot = p_da + p_evidence + p_relab + plot_layout(design = my_layout) + plot_annotation(tag_levels = 'A')
)

## AMP & species abundance

### venn: AMP potential source organism vs species abundance results

- overlap between AMP potential source organism and age-related but gender-irrelevant species (based on abundance)

In [575]:
# age-related but gender-irrelevant species (based on abundance)
pdata <- list(
    `AMP Potential Source Species\n(Also Detected in MetaGenome)\n` = read_tsv('../tables/select-amp-poteintial-source-by_blast-detected.tsv') %>% 
        pull(clade_short),
    `Longivity Relevant Species\n(MetaGenome)\n` = c(
        'Acidaminococcus_intestini', 'Anaerostipes_hadrus', 'Collinsella_aerofaciens',
        'Coprococcus_comes', 'Dialister_sp_CAG_357', 'Eubacterium_siraeum', 
        'Faecalibacterium_prausnitzii', 'Fusicatenibacter_saccharivorans', 'Ruminococcus_gnavus'
    )
)

color_map <- c("#464a96", "#4DAF4A")
names(color_map) <- names(pdata)

In [607]:
pdf(file = file.path(wkdir, 'venn-amp-vs-abundance_result.pdf'), width = 7, height = 5)
plot(
    euler(pdata, shape = 'ellipse'),
    quantities = list(type=c('counts'), cex=1.5),
    fill = color_map, alpha = 0.65,
    # legend = list(side='right', font=1),
    labels = list(font=1)
)
dev.off()

png 
  2

In [609]:
intersect(pdata[[1]], pdata[[2]])

[1] "Acidaminococcus_intestini"    "Eubacterium_siraeum"         
[3] "Coprococcus_comes"            "Anaerostipes_hadrus"         
[5] "Faecalibacterium_prausnitzii"

### volcano show shared species in abundance DA results

In [42]:
shared_species <- c(
    'Acidaminococcus_intestini', 'Eubacterium_siraeum', 'Coprococcus_comes',
    'Anaerostipes_hadrus', 'Faecalibacterium_prausnitzii'
)
status_col = c(change_col, 'unchange'='gray')
p_threshold = 0.05

In [43]:
# volcano plot
pdata <- read_csv('../tables/abundance-da-species.csv') %>% 
    extract(col = clade_name, into = 'clade_short', regex = '.+\\|s__(.+)') %>%
    mutate(status = factor(status, levels = names(status_col)))

p <- ggplot(pdata, aes(x = log2fc, y = -log10(p.adj), fill = status, size = abs(diff))) + 
    geom_point(alpha = 0.7, shape = 21) + 
    geom_hline(yintercept = -log10(p_threshold), linetype = 'dashed', color = 'red') + 
    geom_label_repel(
        data = subset(pdata, clade_short %in% shared_species), 
        aes(label = clade_short), fill = '#ffffffaa', show.legend = F, 
        box.padding = 0.7, size = 4, min.segment.length = 0.001
    ) +
    scale_fill_manual(values = status_col)  +
    labs(x='Log2(Fold Change)', y='-Log10(Adjusted P-value)', size='Abs(Diff)', fill='Abundance\nChange') +
    guides(fill=guide_legend(override.aes = list(size=3))) +
    theme(legend.position = 'right')
ggsave(filename = file.path(wkdir, 'volcano-abundance-da-with-shared-species.pdf'), plot = p, width = 9, height = 6)

## 251202 updates

- update figures based on SLR gender results (`slr-gender`) rather than mlr results

In [3]:
df_slr_age <- read_csv('../tables/abundance-slr-age.csv')
df_slr_gender <- read_csv('../tables/abundance-slr-genderM.csv')

### SLR results: butterfly plot

In [4]:
clade_order <- df_slr_age %>%
    arrange(desc(da.status), desc(clade_short)) %>%
    pull(clade_short)

color_map <- c(
    'Positive' = change_col[['up']],
    'Negative' = change_col[['down']],
    'Irrelevant' = 'gray'
)

In [5]:
pdata <- bind_rows(
    df_slr_age %>% mutate(slr_var = 'SLR: Relative Abundance ~ Age'),
    df_slr_gender %>% mutate(slr_var = 'SLR: Relative Abundance ~ Gender (M)')
) %>%
    mutate(
        slr.status = stringr::str_to_title(slr.status),
        clade_short = factor(clade_short, levels = clade_order)
    )

In [6]:
p <- butterfly_plot(
    plot_data = pdata, x = 'r.squared',y = 'clade_short', 
    left_wing = 'SLR: Relative Abundance ~ Age',
    right_wing = 'SLR: Relative Abundance ~ Gender (M)',
    wing_col = 'slr_var', fill='slr.status'
) &
    scale_fill_manual(values = color_map) &
    labs(x = 'R-Squared', fill = 'Correlation\nCategory') & 
    theme(plot.title = element_text(hjust = 0.5))
ggsave(plot=p, filename = file.path(wkdir, 'slr-r2-age-and-gender-butterflyplot.pdf'), width = 12, height = 5)

### venn: AMP potential source organism vs species abundance results

- overlap between AMP potential source organism and selected species based on abundance data through SLR
- SLR filtering:
  1. gender irrelevant
  2. Case abundance increasing species (up) && not positively correlated to age; Or Case descreasing species (down) && not negatively correlated to age

In [12]:
# abundance species
species_age <- df_slr_age %>% 
    filter(
        ((da.status == 'up') & (slr.status != 'positive')) | 
        ((da.status == 'down') & (slr.status != 'negative')) 
    ) %>% 
    pull(clade_short)
species_gender <- df_slr_gender %>% 
    filter(slr.status == 'irrelevant') %>% 
    pull(clade_short)

# AMP source species
species_amp <- read_tsv('../tables/select-amp-poteintial-source-by_blast-detected.tsv') %>% 
    pull(clade_short)

In [16]:
# age-related but gender-irrelevant species (based on abundance)
pdata <- list(
    `AMP Potential Source Species\n(Also Detected in MetaGenome)\n` = species_amp,
    `Longivity Relevant Species\n(MetaGenome)\n` = intersect(species_age, species_gender)
)

color_map <- c("#464a96", "#4DAF4A")
names(color_map) <- names(pdata)

In [17]:
pdf(file = file.path(wkdir, 'venn-amp-vs-abundance_result-251202.pdf'), width = 7, height = 5)
plot(
    euler(pdata, shape = 'ellipse'),
    quantities = list(type=c('counts'), cex=1.5),
    fill = color_map, alpha = 0.65,
    # legend = list(side='right', font=1),
    labels = list(font=1)
)
dev.off()

png 
  2

In [20]:
print(intersect(pdata[[1]], pdata[[2]]))

 [1] "Eubacterium_sp_CAG_180"            "Eubacterium_sp_CAG_251"           
 [3] "Prevotella_sp_CAG_520"             "Prevotella_copri"                 
 [5] "Bifidobacterium_adolescentis"      "Acidaminococcus_intestini"        
 [7] "Oscillibacter_sp_57_20"            "Bifidobacterium_pseudocatenulatum"
 [9] "Eubacterium_siraeum"               "Coprococcus_comes"                
[11] "Anaerostipes_hadrus"               "Bacteroides_fragilis"             
[13] "Faecalibacterium_prausnitzii"     


In [21]:
write(
    x = intersect(pdata[[1]], pdata[[2]]), 
    sep = '\n',
    file = file.path(wkdir, 'amp-abundance-shared-species-251202.list')
)

### volcano show shared species in abundance DA results

In [49]:
shared_species <- read.csv(file.path(wkdir, 'amp-abundance-shared-species-251202.list'), header=F)$V1
length(shared_species)

color_map = c(change_col, 'unchange'='gray')
p_threshold = 0.05


pdata <- read_csv('../tables/abundance-da-species.csv') %>% 
    extract(col = clade_name, into = 'clade_short', regex = '.+\\|s__(.+)') %>%
    mutate(status = factor(status, levels = names(color_map)))

[1] 13

In [77]:
# volcano plot
set.seed(112)
p <- ggplot(pdata, aes(x = log2fc, y = -log10(p.adj), size = abs(diff))) + 
    geom_point(aes(fill = status), alpha = 0.7, shape = 21) + 
    geom_hline(yintercept = -log10(p_threshold), linetype = 'dashed', color = 'red') + 
    geom_label_repel(
        data = subset(pdata, clade_short %in% shared_species), 
        aes(label = clade_short, color=status), show.legend = F, 
        box.padding = 0.5, size = 3.5, min.segment.length = 0.001,
        fill = '#ffffffaa'
    ) +
    scale_fill_manual(values = color_map)  +
    scale_color_manual(values = color_map)  +
    labs(x='Log2(Fold Change)', y='-Log10(Adjusted P-value)', size='Abs(Diff)', fill='Abundance\nChange') +
    guides(fill=guide_legend(override.aes = list(size=3))) +
    theme(legend.position = 'right')
ggsave(filename = file.path(wkdir, 'volcano-abundance-da-with-shared-species-251202.pdf'), plot = p, width = 9, height = 6)